# PubMed embeddings on Kaggle T4 x2
Run this notebook only with **Save Version > Save & Run All**. It embeds the complete validated 159-file snapshot, verifies every part and checksum, then packages the result into downloadable bundles under `/kaggle/working/pubmed-embeddings`. A `DOWNLOAD_READY.json` marker is written only after everything is complete. Do not use Quick Save and do not run the cells manually.


In [ ]:
DATA_ARCHIVE = 'pubmed-full-articles.zip'
MODEL = 'BAAI/bge-small-en-v1.5'
GPU_BATCH = 128
ROWS_PER_PART = 10_000
PARTS_PER_ARCHIVE = 25
MAX_RUNTIME_MINUTES = 660  # Stop cleanly before Kaggle's 12-hour session limit.


In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, 'Select Settings > Accelerator > GPU T4 x2'
print('Using:', [torch.cuda.get_device_name(i) for i in range(2)])
print(f'Working disk free: {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

In [ ]:
# Kaggle normally expands an uploaded ZIP under /kaggle/input. Also accept the original ZIP.
import json
input_root = Path('/kaggle/input')
data_root = Path('/kaggle/working/pubmed-data')
data_root.mkdir(parents=True, exist_ok=True)
zip_matches = list(input_root.rglob(DATA_ARCHIVE))
manifest_matches = list(input_root.rglob('dataset.json'))
if zip_matches:
    assert len(zip_matches) == 1, f'Expected one {DATA_ARCHIVE}; found {zip_matches}'
    with zipfile.ZipFile(zip_matches[0]) as archive:
        archive.extractall(data_root)
    source_manifest = data_root / 'dataset.json'
    project = data_root
else:
    assert len(manifest_matches) == 1, f'Expected one dataset.json under /kaggle/input; found {manifest_matches}'
    source_manifest = manifest_matches[0]
    project = source_manifest.parent
source_dataset = json.loads(source_manifest.read_text())
source_store = (source_manifest.parent / source_dataset['store']).resolve()
assert (source_store / 'articles.parquet').exists(), source_store / 'articles.parquet'
working_dataset = dict(source_dataset)
working_dataset['store'] = str(source_store)
working_dataset['index'] = 'index'
manifest = data_root / 'dataset.json'
manifest.write_text(json.dumps(working_dataset, indent=2) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'requirements.txt')], check=True)
print('Read-only articles:', source_store / 'articles.parquet')
print('Writable checkpoints:', data_root / 'index')

In [ ]:
# Refuse the old three-file pilot.
import json
manifest = data_root / 'dataset.json'
dataset = json.loads(manifest.read_text())
assert dataset.get('status') == 'ready' and not dataset.get('pilot') and dataset.get('embedding_ready'), dataset
articles = data_root / dataset['store'] / 'articles.parquet'
assert articles.exists(), articles
import hashlib
with articles.open('rb') as source:
    actual_hash = hashlib.file_digest(source, 'sha256').hexdigest()
assert actual_hash == dataset['articles_sha256'], 'Article file checksum mismatch'
os.environ['PUBMED_DATASET'] = str(manifest)
os.environ['PUBMED_EMBED_DEVICE'] = 'cuda:0'
os.environ['PUBMED_EMBED_DEVICES'] = 'cuda:0,cuda:1'
os.environ['PUBMED_ALLOW_MODEL_DOWNLOAD'] = '1'
print('Snapshot:', dataset['snapshot'])
print('Articles:', articles)
print('Article SHA-256 verified:', actual_hash)

## Benchmark both T4 GPUs

In [ ]:
script = project / 'pipeline' / 'embedding_shards.py'
subprocess.run([sys.executable, str(script), 'benchmark', '--dataset', str(manifest),
                '--model', MODEL, '--benchmark-articles', '10000',
                '--gpu-batch', str(GPU_BATCH)], check=True)

## Create all embedding parts
This cell must finish with `stopped_for_time_limit: false`. The next cell refuses incomplete output.


In [ ]:
subprocess.run([sys.executable, str(script), 'worker', '--dataset', str(manifest),
                '--model', MODEL, '--worker', '0', '--workers', '1',
                '--rows-per-part', str(ROWS_PER_PART), '--gpu-batch', str(GPU_BATCH),
                '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES)], check=True)

## Verify, package, and mark the saved output ready
This checks every vector and passage checksum before packaging. It creates about 13 bundles so the laptop can download and extract one at a time.


In [ ]:
import hashlib

verify_run = subprocess.run(
    [sys.executable, str(script), 'verify', '--dataset', str(manifest),
     '--workers', '1', '--rows-per-part', str(ROWS_PER_PART)],
    check=True, capture_output=True, text=True,
)
print(verify_run.stderr, end='')
print(verify_run.stdout, end='')
verified = json.loads(verify_run.stdout)
assert verified['complete'], verified
assert verified['articles'] == dataset['counts']['articles'], verified

index_root = data_root / 'index'
bundle_root = Path('/kaggle/working/pubmed-embeddings')
bundle_root.mkdir(parents=True, exist_ok=True)
(bundle_root / 'source-dataset.json').write_text(
    json.dumps(source_dataset, indent=2) + '\n', encoding='utf-8'
)
plan_source = index_root / 'embedding-checkpoints' / 'w1-0' / 'plan.json'
assert plan_source.is_file(), plan_source
shutil.copy2(plan_source, bundle_root / 'embedding-plan.json')
plan = json.loads(plan_source.read_text(encoding='utf-8'))

part_manifests = [Path(value) for value in verified['manifests']]
archives = []
for start in range(0, len(part_manifests), PARTS_PER_ARCHIVE):
    group = part_manifests[start:start + PARTS_PER_ARCHIVE]
    number = start // PARTS_PER_ARCHIVE
    name = f'embedding-parts-{number:03d}.zip'
    target = bundle_root / name
    temporary = target.with_suffix('.zip.partial')
    members = []
    for part_manifest in group:
        record = json.loads(part_manifest.read_text(encoding='utf-8'))
        members.extend([
            index_root / record['passage_file'],
            index_root / record['vector_file'],
            part_manifest,
        ])
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for member in members:
            assert member.is_file(), member
            archive.write(member, member.relative_to(index_root).as_posix())
    with zipfile.ZipFile(temporary) as archive:
        assert archive.testzip() is None, f'Corrupt archive: {name}'
    os.replace(temporary, target)
    with target.open('rb') as source:
        archive_hash = hashlib.file_digest(source, 'sha256').hexdigest()
    archives.append({
        'name': name,
        'sha256': archive_hash,
        'bytes': target.stat().st_size,
        'first_part': int(group[0].stem.split('-')[-1]),
        'last_part': int(group[-1].stem.split('-')[-1]),
        'parts': len(group),
    })
    for member in members:
        member.unlink()
    print(f'Packaged {len(group)} parts as {name}: {target.stat().st_size / 1e9:.2f} GB', flush=True)

output_manifest = {
    'format': 'pubmed-embedding-bundle-v1',
    'snapshot': verified['snapshot'],
    'descriptor_hash': verified['descriptor_hash'],
    'model': plan['model'],
    'revision': plan['revision'],
    'chunk_version': plan['chunk_version'],
    'vector_dtype': plan['vector_dtype'],
    'dimensions': plan['dimensions'],
    'normalized': plan['normalized'],
    'workers': verified['workers'],
    'rows_per_part': verified['rows_per_part'],
    'articles': verified['articles'],
    'passages': verified['passages'],
    'parts': len(part_manifests),
    'vector_bytes': verified['vector_bytes'],
    'passage_bytes': verified['passage_bytes'],
    'archive_bytes': sum(item['bytes'] for item in archives),
    'archives': archives,
}
output_manifest_path = bundle_root / 'OUTPUT_MANIFEST.json'
output_manifest_path.write_text(json.dumps(output_manifest, indent=2) + '\n', encoding='utf-8')
with output_manifest_path.open('rb') as source:
    output_manifest_hash = hashlib.file_digest(source, 'sha256').hexdigest()
ready = {
    'format': output_manifest['format'],
    'snapshot': output_manifest['snapshot'],
    'articles': output_manifest['articles'],
    'passages': output_manifest['passages'],
    'parts': output_manifest['parts'],
    'archives': len(archives),
    'manifest_sha256': output_manifest_hash,
    'ready': True,
}
Path('/kaggle/working/DOWNLOAD_READY.json').write_text(
    json.dumps(ready, indent=2) + '\n', encoding='utf-8'
)
shutil.rmtree(index_root, ignore_errors=True)
print(json.dumps(ready, indent=2))
print(f'Persistent output size: {sum(item["bytes"] for item in archives) / 1e9:.2f} GB')
print('SUCCESS: Save & Run All can now publish these files as this notebook version output.')


## Finished
The saved version is valid only when its **Output** tab contains `DOWNLOAD_READY.json`, `pubmed-embeddings/OUTPUT_MANIFEST.json`, and every numbered embedding bundle. The laptop watcher verifies these files before downloading anything.
